In [ ]:
from collections import defaultdict
import string
from gensim.models import FastText
from gensim.test.utils import common_texts
import random
import math
import torch
import numpy as np

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from model import load_sentence_encoder, load_model, generate_response, generate_batch
from scoring import clean_response, compute_stability, is_uncertainty_response, plateau_stop, responses_are_substantive
from data import load_triviaqa, load_mmlu, load_webquestions
from evaluate import evaluate, run_full_evaluation, print_comparison_table
from calibration import compute_qhat, check_coverage, run_calibration
from sampler import adaptive_sample, check_coverage, build_prediction_set
from helpers import _assign_split, _print_split_counts, filter_split, clear_embed_cache, embed_cached
from scoring import clean_response, compute_stability, plateau_stop, is_uncertainty_response, responses_are_substantive, token_f1
from config import QASample, SamplerConfig, UNCERTAINTY_TOKENS, COVERAGE_SIM_THRESHOLD, MMLU_SUBJECTS

In [ ]:
model, tokenizer = load_model()

triviaqa_samples   = load_triviaqa(SEED, n_total=3000)
webq_samples       = load_webquestions(SEED)
mmlu_samples       = load_mmlu(SEED)

DATASETS = {
    "triviaqa": triviaqa_samples,
    "webq":     webq_samples,
    "mmlu":     mmlu_samples,
}

# Quick sanity check
for name, samples in DATASETS.items():
    cal  = len(filter_split(samples, "calibration"))
    val  = len(filter_split(samples, "validation"))
    test = len(filter_split(samples, "test"))
    print(f"{name}: cal={cal}, val={val}, test={test}")

In [ ]:
print("Loading FastText model (author's semantic similarity)...")
similarity_model = FastText(
    sentences=common_texts,
    vector_size=200,
    min_count=1,
    epochs=10,
)
print("FastText loaded.")

In [ ]:
def new_CP_score(dict_of_freq, weight, weight_2):
    """Author's nonconformity score. Copied verbatim, only added weight_2 param."""
    dict_of_score = dict_of_freq.copy()
    total_frequency = sum(dict_of_freq.values())

    numerator = 0
    for key, value in dict_of_score.items():
        numerator += -value / total_frequency * math.log(value / total_frequency)
    if total_frequency <= 1:
        total_frequency = 2
    normalized_entropy = numerator / math.log(total_frequency)

    rank_1_response = ""
    for rank, (key, value) in enumerate(dict_of_score.items()):
        if rank == 0:
            rank_1_response = key
            dict_of_score[key] = (10 - value / total_frequency * 10
                                  + normalized_entropy / 2 * weight)
        else:
            dict_of_score[key] = (10 - value / total_frequency * 10
                                  + normalized_entropy / 2 * weight)
            try:
                sim = similarity_model.wv.similarity(key, rank_1_response)
            except KeyError:
                sim = 0.0
            dict_of_score[key] -= sim * weight_2

    return dict_of_score, normalized_entropy


def calculate_quantile_pos(n, alpha):
    return np.ceil((n + 1) * (1 - alpha)) / n


def remove_punctuation(s):
    return s.translate(str.maketrans("", "", string.punctuation))

def remove_articles(s):
    return ' '.join(w for w in s.split() if w.lower() not in {'a', 'an', 'the'})

def remove_duplicate_whitespace(s):
    return ' '.join(s.split())

def normalize(s):
    return remove_duplicate_whitespace(remove_articles(remove_punctuation(s.lower())))

def normalize_dict(d):
    """Merge keys that normalize to the same string."""
    out = {}
    for k, v in d.items():
        nk = normalize(k)
        out[nk] = out.get(nk, 0) + v
    return out


In [ ]:
def generate_lofreefcp_responses(question: str, n: int = 20) -> dict:
    """
    Generate n responses for a question using Mistral.
    Returns frequency dict {normalized_response: count} — same format
    as the author's generation script output.

    Uses temperature=0.9 (same as our AdaptiveCP) for fair comparison.
    """
    freq = defaultdict(int)
    for _ in range(n):
        raw = generate_response(model, tokenizer, question, temperature=0.9)
        cleaned = normalize(clean_response(raw))
        if cleaned:
            freq[cleaned] += 1
    return dict(sorted(freq.items(), key=lambda x: x[1], reverse=True))

In [ ]:
def pregenerate_dataset(dataset_name: str, max_cal: int = 500,
                         max_test: int = 500, sampling_num: int = 20) -> dict:
    """
    Pre-generate all responses once so we don't regenerate for each
    weight combination. Saves ~6000x compute vs the author's approach.

    Returns {
        "cal":  [(gold_normalized, freq_dict), ...],
        "test": [(gold_normalized, freq_dict), ...],
    }
    """
    cal_samples  = filter_split(DATASETS[dataset_name], "calibration")
    test_samples = filter_split(DATASETS[dataset_name], "test")

    if len(cal_samples)  > max_cal:  cal_samples  = random.sample(cal_samples,  max_cal)
    if len(test_samples) > max_test: test_samples = random.sample(test_samples, max_test)

    print(f"\nPre-generating {dataset_name}: "
          f"{len(cal_samples)} cal + {len(test_samples)} test "
          f"× {sampling_num} samples each")
    print(f"Total model calls: "
          f"{(len(cal_samples) + len(test_samples)) * sampling_num}")

    cal_data  = []
    test_data = []

    for i, sample in enumerate(cal_samples):
        freq = generate_lofreefcp_responses(sample.question, n=sampling_num)
        gold = normalize(sample.gold_answers[0]) if sample.gold_answers else ""
        cal_data.append((gold, sample.gold_answers, freq))
        if (i + 1) % 50 == 0:
            print(f"  cal [{i+1}/{len(cal_samples)}]")

    for i, sample in enumerate(test_samples):
        freq = generate_lofreefcp_responses(sample.question, n=sampling_num)
        gold = normalize(sample.gold_answers[0]) if sample.gold_answers else ""
        test_data.append((gold, sample.gold_answers, freq))
        if (i + 1) % 50 == 0:
            print(f"  test [{i+1}/{len(test_samples)}]")

    print(f"Pre-generation complete.")
    return {"cal": cal_data, "test": test_data}

In [ ]:
def run_lofreecp_single(cal_data, test_data, weight, weight_2, alpha):
    """
    Author's exact calibration + evaluation procedure for one parameter set.
    Returns (ECR, SSC, APSS, Abstain%).
    """

    # --- Calibration: compute nonconformity scores ---
    nonconformity_scores = []

    for gold_norm, gold_aliases, freq in cal_data:
        result, _ = new_CP_score(freq, weight, weight_2)

        if gold_norm not in result:
            alias_norms = [normalize(a) for a in gold_aliases]
            found = next((a for a in alias_norms if a in result), None)
            if found:
                nonconformity_scores.append(result[found])
            else:
                nonconformity_scores.append(20)
        else:
            nonconformity_scores.append(result[gold_norm])

    # --- Compute quantile threshold ---
    n = len(nonconformity_scores)
    quantile_pos = calculate_quantile_pos(n, alpha) * 100
    q_val = np.percentile(
        sorted(nonconformity_scores, reverse=True),
        quantile_pos
    )

    # --- Test evaluation ---
    n_covered = 0
    n_ssc_covered = 0
    n_ssc_total = 0
    total_set_size = 0
    n_abstained = 0  # ✅ NEW

    for gold_norm, gold_aliases, freq in test_data:
        result, _ = new_CP_score(freq, weight, weight_2)
        pred_set = [k for k, s in result.items() if s <= q_val]

        alias_norms = {normalize(a) for a in gold_aliases}
        alias_norms.add(gold_norm)

        # --- Coverage check ---
        covered = False
        for pred in pred_set:
            for g in alias_norms:
                if token_f1(pred, g) >= 0.5:
                    covered = True
                    break
            if covered:
                break

        abstained = len(pred_set) == 0

        if abstained:
            n_abstained += 1  # ✅ NEW

        if covered or abstained:
            n_covered += 1

        if not abstained:
            n_ssc_total += 1
            if covered:
                n_ssc_covered += 1

        total_set_size += len(pred_set)

    n_test = len(test_data)

    ecr = n_covered / n_test * 100
    ssc = (n_ssc_covered / n_ssc_total * 100) if n_ssc_total > 0 else 0.0
    apss = total_set_size / n_test
    abstain_rate = n_abstained / n_test * 100  # ✅ NEW

    return ecr, ssc, apss, abstain_rate


In [ ]:
def run_lofreecp_full(dataset_name: str, alpha: float,
                     pregenerated: dict,
                     weight_grid=None) -> dict:
    """
    Full procedure:
    - Grid search over (weight, weight_2)
    - Pick best params by APSS on calibration set
    - Evaluate on test set
    """

    if weight_grid is None:
        weight_grid = [0, 0.4, 0.8, 1.2, 1.6, 2.0]

    cal_data = pregenerated["cal"]
    test_data = pregenerated["test"]

    best_apss = float("inf")
    best_w = (0, 0)

    print(f"  Grid search {len(weight_grid) ** 2} combinations...")

    # --- Grid search ---
    for w1 in weight_grid:
        for w2 in weight_grid:
            ecr, ssc, apss, _ = run_lofreecp_single(
                cal_data, cal_data, w1, w2, alpha
            )

            if apss < best_apss:
                best_apss = apss
                best_w = (w1, w2)

    print(
        f"  Best weights: w1={best_w[0]}, w2={best_w[1]} "
        f"→ cal APSS={best_apss:.3f}"
    )

    # --- Final evaluation ---
    ecr, ssc, apss, abstain_rate = run_lofreecp_single(
        cal_data, test_data, best_w[0], best_w[1], alpha
    )

    return {
        "dataset": dataset_name,
        "alpha": alpha,
        "ECR": round(ecr, 1),
        "SSC": round(ssc, 1),
        "APSS": round(apss, 2),
        "Abstain%": round(abstain_rate, 1),  # ✅ NEW
        "API": 20,
        "best_w1": best_w[0],
        "best_w2": best_w[1],
    }

In [ ]:
import json

ALPHAS_TRIVIAQA = [0.25, 0.30, 0.35, 0.40, 0.45]
ALPHAS_WEBQ     = [0.35, 0.40, 0.45, 0.50]


def run_lofreecp_and_save(dataset_name: str, alphas: list, output_file: str):

    print(f"\n{'='*65}")
    print(f"RUNNING LOFREECP: {dataset_name.upper()}")
    print(f"{'='*65}")

    # Pre-generate once
    pregenerated = pregenerate_dataset(dataset_name, max_cal=500, max_test=500)

    results = []

    for alpha in alphas:
        print(f"\nRunning LofreeCP on {dataset_name} | alpha={alpha}...")

        result = run_lofreecp_full(dataset_name, alpha, pregenerated)

        print(
            f"  ECR={result['ECR']}  SSC={result['SSC']}  "
            f"APSS={result['APSS']}  Abstain%={result['Abstain%']}"
        )

        results.append(result)

    # --- Save to file ---
    with open(output_file, "w") as f:
        json.dump(results, f, indent=4)

    print(f"\nSaved results to: {output_file}")

    return results


# Run and save
lofree_triviaqa = run_lofreecp_and_save(
    "triviaqa",
    ALPHAS_TRIVIAQA,
    "lofreecp_triviaqa_results.json"
)

lofree_webq = run_lofreecp_and_save(
    "webq",
    ALPHAS_WEBQ,
    "lofreecp_webq_results.json"
)